# Set up in cloud

After running this cell, restart the notebook session. TODO: activate venv instead

In [2]:
!pip install -e .

Obtaining file:///home/jupyter/grouping-trainer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 135.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 124.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 122.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 38.6 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 51.1 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 143.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 117.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 65.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 53.8 MB/s  0:00:06m0:

In [3]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

Copying gs://seer-models/models/issue_grouping_v1/.DS_Store...
Copying gs://seer-models/models/issue_grouping_v1/data.pkl...                   
Copying gs://seer-models/models/issue_grouping_v1/embeddings/1_Pooling/config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/README.md...       
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config.json...     
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config_sentence_transformers.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/configuration_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/merges.txt...      
Copying gs://seer-models/models/issue_grouping_v1/embeddings/model.safetensors...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modeling_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modules.json...    
Copying gs://seer-models/models/issue_grouping_v1/embeddings/special_tokens_map.json...
Copying gs:

In [4]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

Copying gs://grouping-data/final_csvs/synthetic-semi-easy-negatives.csv...
Copying gs://grouping-data/final_csvs/test.csv...                               
Copying gs://grouping-data/final_csvs/train.csv...                              
Copying gs://grouping-data/final_csvs/val.csv...                                
| [4/5 files][  5.4 GiB/  5.4 GiB]  99% Done 236.4 MiB/s ETA 00:00:00           

# Set up

In [1]:
token_value = !gcloud secrets versions access latest --secret=wandb-api-key --project=996102297610
import os

os.environ["WANDB_API_KEY"] = token_value[0]
del token_value

In [2]:
import wandb

For some reason, you need to run this next cell, interrupt it (it will hang), and then run it again (it will
immediately succeed)

In [4]:
wandb.login()

wandb: Currently logged in as: kush-dubey (sentry-seer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import math
import warnings

from datetime import datetime
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import MultiDatasetBatchSamplers
import torch

import grouping_trainer as gt
import utils

In [6]:
assert torch.cuda.is_available()

In [7]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

Some vars to care about

In [14]:
RUN_SHORTNAME = "gte"

SAMPLE_TRAIN: int | None = None if torch.cuda.is_available() else 30
SAMPLE_VAL: int | None = 5000 if torch.cuda.is_available() else 20

PER_DEVICE_TRAIN_BATCH_SIZE = 256 if torch.cuda.is_available() else 2
GRADIENT_ACCUMULATION_STEPS = 1
GRADIENT_CHECKPOINTING = not torch.cuda.is_available()  # turn off if model supports flash attn

PER_DEVICE_EVAL_BATCH_SIZE = 2
EVAL_STEPS = 300
PER_DEVICE_TOKEN_BUDGET = 8192 * 4  # increased for A100 80GB

OUTPUT_DIR = f"./{timestamp}-{RUN_SHORTNAME}-output"

In [9]:
assert (EVAL_STEPS % 5) == 0, "pls for sanity make it divisible by 5"

# Load model

In [10]:
gt.utils._cuda_empty_cache()

In [11]:
# model_path = "issue_grouping_v1/embeddings"
# # model_path = "/Users/kdubey/projects/seer/models/issue_grouping_v1/embeddings"
# model = gt.utils.SentenceTransformer(
#     str(model_path),
#     trust_remote_code=True,
#     # model_kwargs=dict(
#     #     dtype=torch.bfloat16,
#     #     attn_implementation="sdpa",  # not possible for jina-ai :-(
#     # )
# )
# model.device

In [11]:
model = gt.utils.SentenceTransformer(
    "Alibaba-NLP/gte-modernbert-base",
    model_kwargs=dict(
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )
)

In [12]:
assert "layernorm" in repr(model[0].auto_model).lower()
assert "batch" not in repr(model[0].auto_model).lower()

Don't have batch norm. That could mess up stuff for the deduplication strategy.

In [13]:
_ = model.encode("test")

# Load data

We'll make a `dataset_val` for val loss.

In [15]:
dataset_val = gt.train.df_to_dataset(utils.load_val_df(sample_size=SAMPLE_VAL))
len(dataset_val)

5000

In [16]:
dataset_dict_train, frac_positive = utils.load_train_dataset_dict(
    sample_size=SAMPLE_TRAIN, min_dataset_size=PER_DEVICE_TRAIN_BATCH_SIZE
)
len(dataset_dict_train)

  0%|          | 0/151 [00:00<?, ?it/s]

133

In [17]:
sum(dataset_dict_train.num_rows.values())

835765

# Set up `Trainer`

In [18]:
evaluator = gt.evaluator.MinPrecisionEvaluator(
    sentences1=list(dataset_val["query_stacktrace_string"]),
    sentences2=list(dataset_val["candidate_stacktrace_string"]),
    labels=[int(record["label"]) for record in dataset_val],
    name="val",
    show_progress_bar=True,
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    truncate_dims=(64, 768),
)

Before training:

In [19]:
evaluator(model)

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

{'val_dim64_pr85_threshold': 0.9023020267486572,
 'val_dim64_pr85_precision': 0.8500404203718674,
 'val_dim64_pr85_recall': 0.6076278532216123,
 'val_dim64_pr85_n_predictions': 2474.0,
 'val_dim64_pr90_threshold': 0.9314619302749634,
 'val_dim64_pr90_precision': 0.9001692047377327,
 'val_dim64_pr90_recall': 0.4611383993065588,
 'val_dim64_pr90_n_predictions': 1773.0,
 'val_dim64_pr95_threshold': 0.9605696797370911,
 'val_dim64_pr95_precision': 0.9507186858316222,
 'val_dim64_pr95_recall': 0.2675527304247327,
 'val_dim64_pr95_n_predictions': 974.0,
 'val_dim64_pr99_threshold': 0.993012011051178,
 'val_dim64_pr99_precision': 0.9916666666666667,
 'val_dim64_pr99_recall': 0.034383126264085524,
 'val_dim64_pr99_n_predictions': 120.0,
 'val_dim768_pr85_threshold': 0.9297424554824829,
 'val_dim768_pr85_precision': 0.8501362397820164,
 'val_dim768_pr85_recall': 0.6310314937879226,
 'val_dim768_pr85_n_predictions': 2569.0,
 'val_dim768_pr90_threshold': 0.9517685174942017,
 'val_dim768_pr90_prec

In [20]:
def init_bias(frac_positive: float):
    return math.log(frac_positive / (1 - frac_positive))

In [21]:
trainer = gt.train.Trainer(
    model=model,
    args=SentenceTransformerTrainingArguments(
        # These should prolly be unchanged
        output_dir=OUTPUT_DIR,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=False,
        dataloader_pin_memory=torch.cuda.is_available(),
        num_train_epochs=1,
        # Save memory
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        #
        # Datalaoder
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
        # Each iter, pick a project randomly, sample from it.
        # Next iter, pick another project randomly, sample from it, etc.
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        seed=42,  # passed to batch sampler
        #
        # Optimizer
        learning_rate=1e-4,
        learning_rate_mapping={
            # These are important to tune. Higher so that training doesn't get stuck. TODO: check
            r"^log_scale$": 2e-4,
            r"^bias$": 2e-4,
        },
        weight_decay=0.01,
        warmup_ratio=0.1,
        #
        # Eval
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        #
        # Logging
        logging_strategy="steps",
        logging_steps=EVAL_STEPS // 10,  # train loss alongside metrics table
        run_name=f"{timestamp}-{RUN_SHORTNAME}",
        report_to="wandb",
        #
        # Checkpointing
        save_strategy="steps",
        save_steps=EVAL_STEPS // 2,
        save_total_limit=2,
    ),
    #
    # Training
    loss=gt.train.SigmoidPairwiseLoss(
        model,
        bias_init=init_bias(frac_positive),
        log_of_scale_init=torch.tensor(5).log(),
        matryoshka_dims=[768, 512, 256, 128, 64],
        matryoshka_weights=[2, 1, 1, 0.5, 0.25],
        n_dims_per_step=2,
    ),
    data_collator=gt.train.DefaultDataCollator(tokenize_fn=model.tokenize),
    train_dataset=dataset_dict_train,
    shuffle_within_dataset=False,  # more cache hits in each forward
    per_device_token_budget=PER_DEVICE_TOKEN_BUDGET,
    #
    # Evaluator
    eval_dataset=dataset_val,  # val loss
    evaluator=evaluator,  # val recall at x precision
)

In [22]:
warnings.filterwarnings(
    "ignore",
    message=".*torch.utils.checkpoint: the use_reentrant parameter.*",
    category=UserWarning,
)

In [ ]:
train_output = trainer.train()

Step,Training Loss,Validation Loss,Val Dim64 Pr85 Threshold,Val Dim64 Pr85 Precision,Val Dim64 Pr85 Recall,Val Dim64 Pr85 N Predictions,Val Dim64 Pr90 Threshold,Val Dim64 Pr90 Precision,Val Dim64 Pr90 Recall,Val Dim64 Pr90 N Predictions,Val Dim64 Pr95 Threshold,Val Dim64 Pr95 Precision,Val Dim64 Pr95 Recall,Val Dim64 Pr95 N Predictions,Val Dim64 Pr99 Threshold,Val Dim64 Pr99 Precision,Val Dim64 Pr99 Recall,Val Dim64 Pr99 N Predictions,Val Dim768 Pr85 Threshold,Val Dim768 Pr85 Precision,Val Dim768 Pr85 Recall,Val Dim768 Pr85 N Predictions,Val Dim768 Pr90 Threshold,Val Dim768 Pr90 Precision,Val Dim768 Pr90 Recall,Val Dim768 Pr90 N Predictions,Val Dim768 Pr95 Threshold,Val Dim768 Pr95 Precision,Val Dim768 Pr95 Recall,Val Dim768 Pr95 N Predictions,Val Dim768 Pr99 Threshold,Val Dim768 Pr99 Precision,Val Dim768 Pr99 Recall,Val Dim768 Pr99 N Predictions
300,0.301500,0.401838,0.382305,0.850014,0.880959,3587.000000,0.521836,0.900098,0.793990,3053.000000,0.723632,0.950308,0.580179,2113.000000,0.943891,0.990991,0.158914,555.000000,0.363328,0.850014,0.895695,3647.000000,0.527110,0.900228,0.797746,3067.000000,0.698201,0.950172,0.639122,2328.000000,0.948044,0.990099,0.144467,505.000000
600,0.259400,0.368709,0.293254,0.850180,0.887027,3611.000000,0.432365,0.900032,0.809015,3111.000000,0.620876,0.950041,0.664837,2422.000000,0.886315,0.990558,0.333430,1165.000000,0.261396,0.850175,0.911586,3711.000000,0.399298,0.900218,0.834152,3207.000000,0.586250,0.950157,0.699509,2548.000000,0.859739,0.990525,0.392661,1372.000000
900,0.197100,0.359597,0.315732,0.850068,0.907541,3695.000000,0.455100,0.900000,0.837330,3220.000000,0.643558,0.950259,0.689974,2513.000000,0.860497,0.990476,0.450737,1575.000000,0.278620,0.850013,0.930078,3787.000000,0.431328,0.900000,0.863334,3320.000000,0.627707,0.950175,0.705287,2569.000000,0.853486,0.990172,0.465761,1628.000000
1200,0.184900,0.347004,0.307268,0.850106,0.922566,3756.000000,0.451098,0.900119,0.874892,3364.000000,0.651629,0.950092,0.742560,2705.000000,0.878082,0.990244,0.469229,1640.000000,0.268375,0.850026,0.941635,3834.000000,0.439667,0.900118,0.885293,3404.000000,0.639647,0.950055,0.752962,2743.000000,0.890051,0.990247,0.440046,1538.000000
1500,0.184800,0.353489,0.286304,0.850013,0.910431,3707.000000,0.427903,0.900030,0.861023,3311.000000,0.666986,0.950059,0.703554,2563.000000,0.883789,0.990489,0.421266,1472.000000,0.238217,0.850078,0.938746,3822.000000,0.395964,0.900177,0.883271,3396.000000,0.607319,0.950000,0.763074,2780.000000,0.857039,0.990373,0.475585,1662.000000
1800,0.182000,0.360489,0.311077,0.850067,0.922277,3755.000000,0.463435,0.900151,0.862179,3315.000000,0.656907,0.950095,0.726091,2645.000000,0.905493,0.990113,0.405085,1416.000000,0.279645,0.850078,0.938746,3822.000000,0.439771,0.900118,0.885293,3404.000000,0.617345,0.950286,0.767697,2796.000000,0.862391,0.990401,0.506790,1771.000000
2100,0.203800,0.368186,0.315022,0.850067,0.922277,3755.000000,0.467249,0.900000,0.863334,3320.000000,0.657977,0.950000,0.730136,2660.000000,0.892274,0.990266,0.440913,1541.000000,0.276766,0.850104,0.942213,3836.000000,0.444059,0.900147,0.885582,3405.000000,0.628256,0.950143,0.765386,2788.000000,0.861673,0.990374,0.505345,1766.000000
2400,0.175800,0.374429,0.314744,0.850159,0.924588,3764.000000,0.465475,0.900090,0.869402,3343.000000,0.661421,0.950093,0.737070,2685.000000,0.898525,0.990060,0.431667,1509.000000,0.274824,0.850039,0.943369,3841.000000,0.442818,0.900117,0.890494,3424.000000,0.634135,0.950125,0.770587,2807.000000,0.897634,0.990132,0.434845,1520.000000
2700,0.181900,0.374009,0.310060,0.850093,0.925744,3769.000000,0.461336,0.900120,0.869691,3344.000000,0.657084,0.950074,0.742271,2704.000000,0.904599,0.990358,0.415487,1452.000000,0.273357,0.850182,0.942791,3838.000000,0.442891,0.900058,0.889916,3422.000000,0.629724,0.950071,0.775209,2824.000000,0.875733,0.990521,0.483097,1688.000000
3000,0.206500,0.380852,0.317966,0.850093,0.925744,3769.000000,0.467043,0.900149,0.869980,3345.000000,0.662379,0.950166,0.743716,270

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

Batches:   0%|          | 0/3872 [00:00<?, ?it/s]

In [25]:
trainer.save_model()

In [26]:
!gsutil -m cp -r wandb gs://grouping-data/runs/{OUTPUT_DIR}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Skipping symlink directory "wandb/latest-run"
Copying file://wandb/debug-internal.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_155636-osu9p69o/run-osu9p69o.wandb [Content-Type=application/octet-stream]...
Copying file://wandb/debug.log [Content-Type=application/octet-stream]...       
Copying file://wandb/run-20251219_155636-osu9p69o/files/wandb-metadata.json [Content-Type=application/json]...
Copying file://wandb/run-20251219_155636-osu9p69o/files/output.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_155636-osu9p69o/files/requirements.txt [Content-Type=text/plain]...
Copying file://wandb/run-20251219_155636-osu9p69o/logs/debug-core.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_155636-osu9p69o/logs/debug-internal.log [Content-Type=application/octet-stream]...
Copying file://wandb/run-20251219_155636-osu9p69o/logs/debug.log [Content-Type=application/octet-stream]...
Copying file://wa

In [27]:
!gsutil -m rsync -r {OUTPUT_DIR} gs://grouping-data/runs/{OUTPUT_DIR}/training

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Building synchronization state...
Starting synchronization...
Copying file://./2025-12-19-15-53-06-gte-output/1_Pooling/config.json [Content-Type=application/json]...
Copying file://./2025-12-19-15-53-06-gte-output/checkpoint-3300/1_Pooling/config.json [Content-Type=application/json]...
Copying file://./2025-12-19-15-53-06-gte-output/checkpoint-3300/model.safetensors [Content-Type=application/octet-stream]...
==> NOTE: You are uploading one or more large file(s), which would run          
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computi

In [28]:
!gsutil -m cp -r train.ipynb gs://grouping-data/runs/{OUTPUT_DIR}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Copying file://train.ipynb [Content-Type=application/octet-stream]...
/ [1/1 files][ 78.6 KiB/ 78.6 KiB] 100% Done                                    
Operation completed over 1 objects/78.6 KiB.                                     
